# Social Reward and Trust Across Adulthood

A short lifespan social-neuroscience tutorial. We ask whether younger and older
adults respond differently to **friends, strangers, and computer partners** in a
trust game, then show how the same task structure becomes regressors in an fMRI
GLM. This notebook mirrors `README.md`.

**What you'll do:**
1. Run a no-download trust-game demo and fit an age × partner model.
2. Apply the same code to real OpenNeuro **ds005123** events.
3. Run one first-level **friend > stranger** GLM.

> Part 1 needs only numpy/pandas/scipy/matplotlib. Parts 2-3 add nilearn +
> openneuro-py and a download.

## Part 1 — Behavioral question

> Do people share more with friends than strangers or computers, and does that
> partner effect differ between younger and older adults?

We simulate repeated trust-game trials for three partner types and fit a linear
model `amount_shared ~ age_group + partner + age_group:partner`. The interaction
terms are the lifespan part — they test whether social context shifts with age.

In [ ]:
import sys
from pathlib import Path

code_dir = Path('code').resolve()
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))
import social_reward_trust as srt

df = srt.simulate_trust_data()      # 50 participants x 3 partners x trials
df.head()

In [ ]:
# Fit the age x partner model and summarize behavior
model = srt.fit_linear_model(df)
print(srt.summarize_by_group(df).to_string(index=False))
model

In [ ]:
# Plot the demo (younger vs older across partners) and show it inline
out = srt.RESULTS_DIR / 'social_reward_trust_demo.png'
srt.plot_demo(df, model, out)
from IPython.display import Image
Image(str(out))

**Read it.** Both groups share more with **friend > stranger > computer**. The
interaction terms (`older_x_friend`, `older_x_stranger`) test whether that gradient
differs by age — the lifespan-social-neuroscience question. Treat the simulated
effect as a scaffold for the method, not a scientific result.

## Part 2 — Real OpenNeuro events

Download ds005123 (or a small subset — event files are tiny):

```bash
pip install openneuro-py
python3 -m openneuro download --dataset ds005123 --target-dir ds005123 \
  --include 'sub-*/func/*task-*trust*_events.*' --include participants.tsv
```

Then summarize the **real** trust behavior with the same model. The script
auto-infers the partner and choice columns from the BIDS events.

In [ ]:
# Run after downloading ds005123; adjust the path as needed.
import importlib; importlib.reload(srt)
srt.run_real_behavior(Path('../ds005123'))

## Part 3 — One first-level GLM

Turn the partner labels into `friend` / `stranger` / `computer` regressors and
map a **friend > stranger** contrast for one participant. Needs a trust-task BOLD
run (preprocessed, or one raw echo) and nilearn.

In [ ]:
# Run after downloading a trust-task BOLD run for one subject.
srt.run_one_glm(Path('../ds005123'), subject='sub-10369')

> **Caveats.** Age-group differences in social reward are subtle and
task-dependent, and a single-subject GLM is a tutorial artifact — not evidence
for age effects. For a real study you would model all runs and subjects, then run
a second-level model for group and age effects.

### You did it
You framed aging as a *social* question, quantified a partner effect, tested an
age-by-partner interaction, and saw how social task structure enters a
reproducible neuroimaging model.